In [182]:
import matplotlib.pyplot as plt
import os
import json
import numpy as np
import csv
import pprint as p
import torch
import pickle
from datasets import Dataset, load_from_disk

# Get all evaluation values

In [183]:
def create_eval_dictionairy(path):
    scores = {}
    for filename in os.listdir(path):
        if filename.endswith(".json"):
            file_path = os.path.join(path, filename)
            with open(file_path, "r") as f:
                data = json.load(f)
                # Get the scores (either mse or f1, only one can be present)
                value = data.get("eval_mse", None)
                metric = "mse"
                if value is None:
                    value = data.get("eval_f1_macro")
                    metric = "f1"
                scores[filename.split("_")[0] + "_" + metric] = value
    return scores

In [184]:
import os
directory = "../eval_results"
subdirs = [os.path.join(directory, d) for d in os.listdir(directory) if os.path.isdir(os.path.join(directory, d))]

eval_dicts = {}
for subdir in subdirs:
    eval_dicts[subdir.split("/")[-1]] = create_eval_dictionairy(subdir)

print(eval_dicts.keys())
print(eval_dicts["Semeval2018Intensity"])

dict_keys(['Semeval2018Intensity', 'WASSA22', 'EmotionStimulus', 'UsVsThem', 'EmoBank', 'SentimentalLIAR', 'CancerEmo', 'TalesEmotions', 'GoodNewsEveryone', 'XED'])
{'WASSA22_mse': 0.04834771901369095, 'TalesEmotions_mse': 0.04647809639573097, 'EmoBank_mse': 0.049344342201948166, 'EmotionStimulus_mse': 0.03820764645934105, 'SentimentalLIAR_mse': 0.04309679940342903, 'CancerEmo_mse': 0.04538495093584061, 'GoodNewsEveryone_mse': 0.043582573533058167, 'XED_mse': 0.04711509495973587, 'Semeval2018Intensity_mse': 0.02593066729605198, 'UsVsThem_mse': 0.05515897274017334}


# Normalise all values
First get max min values across all datasets

In [185]:
mse_values = []
f1_values = []
for dataset_name, metrics in eval_dicts.items():
    for key, value in metrics.items():
        if key.endswith('_mse'):
            mse_values.append(value)
        elif key.endswith('_f1'):
            f1_values.append(value)

min_mse = min(mse_values)
max_mse = max(mse_values)
print(f"Min MSE: {min_mse}, Max MSE: {max_mse}")


min_f1 = min(f1_values)
max_f1 = max(f1_values)
print(f"Min F1: {min_f1}, Max F1: {max_f1}")

Min MSE: 0.007274538278579712, Max MSE: 0.08849239349365234
Min F1: 0.0, Max F1: 0.967240928078629


### Normalise

Normalized MSE = $1-\frac{\text{MSE - min(MSE)}}{max(MSE) - min(MSE)}$

Normalized F1 = $1-\frac{\text{F1 - min(F1)}}{max(F1) - min(F1)}$

In [186]:
normalized_eval_dicts = {}

for dataset_name, metrics in eval_dicts.items():
    normalized_metrics = {}
    for key, value in metrics.items():
        if key.endswith('_mse'):
            normalized_value = 1 - (value - min_mse) / (max_mse - min_mse)
            normalized_metrics[key] = normalized_value
        elif key.endswith('_f1'):
            normalized_value = (value - min_f1) / (max_f1 - min_f1)
            normalized_metrics[key] = normalized_value
    normalized_eval_dicts[dataset_name] = normalized_metrics

In [187]:
p.pprint(normalized_eval_dicts)

{'CancerEmo': {'CancerEmo_f1': 0.25300455899801116,
               'EmoBank_f1': 0.17043950828781274,
               'EmotionStimulus_f1': 0.1946991241798678,
               'GoodNewsEveryone_f1': 0.17301143735343885,
               'Semeval2018Intensity_f1': 0.17423644169567123,
               'SentimentalLIAR_f1': 0.13234368772112462,
               'TalesEmotions_f1': 0.18822640446602457,
               'UsVsThem_f1': 0.05682998509545966,
               'WASSA22_f1': 0.15822301793304588,
               'XED_f1': 0.09653193216437753},
 'EmoBank': {'CancerEmo_mse': 0.29668670664632824,
             'EmoBank_mse': 0.31184750656736215,
             'EmotionStimulus_mse': 0.0,
             'GoodNewsEveryone_mse': 0.19459674616975742,
             'Semeval2018Intensity_mse': 0.44101863923596596,
             'SentimentalLIAR_mse': 0.24244446042685008,
             'TalesEmotions_mse': 0.12197561003354229,
             'UsVsThem_mse': 0.12399535596955691,
             'WASSA22_mse': 0.2257

### Create the groups for oridinal regression
To keep it simple, we use 3 groups (Good, Decent, Bad) each a 33% split of the normalised values
These groups are not to train on, the idea is that we train for the continous normalised value, then put it within the group based on these group values
And then we will see if we performed well

So we will train using MSE loss and then based on the MSE loss we will get a value that should then be put into a group based on these values

So training and evaluation will be done seperately

In [188]:
import numpy as np

# Get all normalised scores to create the groups
all_scores = []

for dataset_name, metrics in normalized_eval_dicts.items():
    for key, value in metrics.items():
        all_scores.append(value)

# Get the quantities that will make the groups
q33 = np.quantile(all_scores, 0.33)
q66 = np.quantile(all_scores, 0.66)

print(f"Group thresholds: 33%={q33:.3f}, 66%={q66:.3f}")

# Assign the groups to the data based on their normalised socre
grouped_eval_dicts = {}

for dataset_name, metrics in normalized_eval_dicts.items():
    grouped_metrics = {}
    for key, value in metrics.items():
        if value <= q33:
            group = 'Bad'
        elif value <= q66:
            group = 'Decent'
        else:
            group = 'Good'
        grouped_metrics[key] = group

    grouped_eval_dicts[dataset_name] = grouped_metrics

p.pprint(grouped_eval_dicts)

Group thresholds: 33%=0.166, 66%=0.393
{'CancerEmo': {'CancerEmo_f1': 'Decent',
               'EmoBank_f1': 'Decent',
               'EmotionStimulus_f1': 'Decent',
               'GoodNewsEveryone_f1': 'Decent',
               'Semeval2018Intensity_f1': 'Decent',
               'SentimentalLIAR_f1': 'Bad',
               'TalesEmotions_f1': 'Decent',
               'UsVsThem_f1': 'Bad',
               'WASSA22_f1': 'Bad',
               'XED_f1': 'Bad'},
 'EmoBank': {'CancerEmo_mse': 'Decent',
             'EmoBank_mse': 'Decent',
             'EmotionStimulus_mse': 'Bad',
             'GoodNewsEveryone_mse': 'Decent',
             'Semeval2018Intensity_mse': 'Good',
             'SentimentalLIAR_mse': 'Decent',
             'TalesEmotions_mse': 'Bad',
             'UsVsThem_mse': 'Bad',
             'WASSA22_mse': 'Decent',
             'XED_mse': 'Bad'},
 'EmotionStimulus': {'CancerEmo_f1': 'Good',
                     'EmoBank_f1': 'Decent',
                     'EmotionStimulus_f

## get all dataset lengths and normalise them for feature representation

In [189]:
data_dir = "../Preprocessed_Data"
dataset_paths = [os.path.join(data_dir, d) for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]

dataset_sizes = {}
for path in dataset_paths:
    dataset_name = os.path.basename(path)
    dataset = load_from_disk(path)
    num_rows = dataset.num_rows
    dataset_sizes[dataset_name] = num_rows

max_size = max(dataset_sizes.values())
min_size = min(dataset_sizes.values())

normalised_sizes = {}
for name, size in dataset_sizes.items():
    normalised = (size - min_size) / (max_size - min_size) if max_size != min_size else 0.0
    normalised_sizes[name] = normalised

print("Dataset Sizes (normalised):")
p.pprint(normalised_sizes)

Dataset Sizes (normalised):
{'CancerEmo': 0.37949331737482545,
 'EmoBank': 0.31645721125074805,
 'EmotionStimulus': 0.011051266706562937,
 'GoodNewsEveryone': 0.11450229403550768,
 'Semeval2018Intensity': 0.35747057650109715,
 'SentimentalLIAR': 0.4251346499102334,
 'TalesEmotions': 0.5255136644723718,
 'UsVsThem': 0.1888689407540395,
 'WASSA22': 0.0,
 'XED': 1.0}


### Get the amount of emotional labels for each dataset

In [190]:
data_dir = "../Preprocessed_Data"
dataset_paths = [os.path.join(data_dir, d) for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]

dataset_emotion_labels = {}

for path in dataset_paths:
        with open(path+"/dataset_info.json") as f:
                info = json.load(f)
                dataset_name = os.path.basename(path)
                dataset_emotion_labels[dataset_name] = info["emotion_labels"]

p.pprint(dataset_emotion_labels)

{'CancerEmo': ['anger',
               'anticipation',
               'disgust',
               'fear',
               'joy',
               'sadness',
               'surprise',
               'trust'],
 'EmoBank': ['V', 'A', 'D'],
 'EmotionStimulus': ['anger',
                     'disgust',
                     'fear',
                     'happy',
                     'sad',
                     'shame',
                     'surprise'],
 'GoodNewsEveryone': ['anger',
                      'annoyance',
                      'disgust',
                      'fear',
                      'guilt',
                      'joy',
                      'love_including_like',
                      'negative_anticipation_including_pessimism',
                      'negative_surprise',
                      'positive_anticipation_including_optimism',
                      'positive_surprise',
                      'pride',
                      'sadness',
                      'shame',
      

# Create the data for training
### For this we also have to load the embeddings (task and text)
We use the cosine similarity between the task embeddings of dataset A-B and the cosine similarity of the text embeddings of dataset A-B

In [191]:
from scipy.spatial.distance import cosine

In [192]:
def find_pt_file(root_dir):
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for file in filenames:
            if file.endswith('.pt'):
                return os.path.join(dirpath, file)
    return None

In [ ]:
# Prepare rows: each row = [dataset_name, metric_name, normalized_value]
rows = []
for dataset_name, metrics in normalized_eval_dicts.items():
    for metric_name, normalized_value in metrics.items():
        source_model = dataset_name
        target_model = metric_name.split("_")[0]
        source_task_embedding = torch.load(find_pt_file(f"../task_embeddings/{source_model}"))
        target_task_embedding = torch.load(find_pt_file(f"../task_embeddings/{target_model}"))

        source_text_embedding = torch.load(find_pt_file(f"../text_embeddings/{source_model}"))
        target_text_embedding = torch.load(find_pt_file(f"../text_embeddings/{target_model}"))

        task_l2_distance = torch.norm(source_task_embedding - target_task_embedding, p=2)
        text_l2_distance = torch.norm(source_text_embedding - target_text_embedding, p=2)

        dataset_size = normalised_sizes[target_model]

        mean_performance = np.mean(list(normalized_eval_dicts["CancerEmo"].values()))

        loss_type = metric_name.split("_")[1]
        if loss_type == 'mse':
            loss_type = 0 # Regression
        else:
            loss_type = 1 # Classification

        label_count_ratio = len(dataset_emotion_labels[target_model]) / len(dataset_emotion_labels[source_model]) # The amount of emotion labels the dataset has

        features = [
            torch.nn.functional.cosine_similarity(source_task_embedding, target_task_embedding, dim=0).item(), 
                    torch.nn.functional.cosine_similarity(source_text_embedding, target_text_embedding, dim=0).item(),
                    task_l2_distance,
                    text_l2_distance,
                    dataset_size,
                    mean_performance,
                    loss_type,
                    label_count_ratio
                    ]

        rows.append({
            'dataset': dataset_name,
            'target_model': target_model,
            'metric': metric_name.split("_")[-1],
            'features': features,
            'normalized_value': normalized_value
        })

        del source_task_embedding, target_task_embedding, source_text_embedding, target_text_embedding
        torch.cuda.empty_cache()

# Save as pkl for easy dataset loading
with open('data.pkl', 'wb') as f:
    pickle.dump(rows, f)

TypeError: unsupported operand type(s) for /: 'list' and 'list'